# ErgoPose Risk Classifier — Model Training

This notebook is the **third stage** of the *ErgoPose Risk Classifier* project.  
It defines, trains, and evaluates the **Artificial Neural Network (ANN)** for posture classification based on the preprocessed dataset.

### Objectives
- Load the cleaned dataset from `data/processed/`.
- Encode categorical posture labels.
- Split the data into training and testing sets.
- Define and train an ANN model for multi-class classification.
- Save the trained model and scaler to the `models/` directory.

### Input and Output
- **Input:** `data/processed/clean_postural_risk_dataset.csv`  
- **Outputs:**  
  - `models/neural_network.pkl`  
  - `models/scaler.pkl`

In [1]:
"""
Imports the necessary libraries for model definition, training, and evaluation.
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import joblib
import json
from math import sqrt

import torch
from torch import Tensor, nn, optim

In [2]:
"""
Defines paths for processed data and model output directories.
"""

DATA_PATH = Path("../data/processed/clean_postural_risk_dataset.csv")
MODELS_PATH = Path("../models")
MODELS_PATH.mkdir(exist_ok=True)

print(f"Dataset path: {DATA_PATH}")
print(f"Models directory: {MODELS_PATH}")


Dataset path: ../data/processed/clean_postural_risk_dataset.csv
Models directory: ../models


In [3]:
"""
Defines the number of neurons in the hidden layers by the 'Geometric Pyramid Rule'
"""

data = pd.read_csv(DATA_PATH)

X = data.drop(columns=['upperbody_label'])
y = data['upperbody_label']

input_neurons = X.shape[1]
output_neurons = 2 # Binary classification

input_output_neurons = int(sqrt(input_neurons*output_neurons))

print(f"The number of neurons in hidden layers will be: {int(input_output_neurons*0.5)} <= N <= {int(input_output_neurons*2)}")

The number of neurons in hidden layers will be: 5 <= N <= 20


In [4]:
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# print(f"Training data shape: {X_train.shape}")
# print(f"Testing data shape: {X_test.shape}")

num_splits = 5
kf = KFold(n_splits=num_splits)


## Model Architecture Proposals - rules
- Hidden layers must have beetwen **5 and 20 neurons total**. If more than 1 hidden layer is implemented, the number of neurons of both layers must **add up** to a number beetwen **5 and 20**.
- Batch size, at this initial stage, must be **default**.
- Activation function **cannot** be tanh.
- Learning rate must be $10^{-2}, 10^{-3}$ or **smaller numbers**.

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [6]:
class MLPModel(nn.Module):
    def __init__(self, input_dim, hidden_dim_1, hidden_dim_2, output_dim):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim_1),
            nn.ReLU(),
            nn.Linear(hidden_dim_1, hidden_dim_2),
            nn.ReLU(),
            nn.Linear(hidden_dim_2, output_dim),
        )

    def forward(self, x):
        return self.net(x)


# Hyperparameters
input_dim: int = X.shape[1]
hidden_dim_1 = 8
hidden_dim_2 = 10
output_dim = 1

X_t: Tensor = torch.tensor(X.values, dtype=torch.float32).to(device)

le = LabelEncoder()
y_enc = le.fit_transform(y)
y_enc = torch.tensor(y_enc, dtype=torch.float32)

acc_scores = []
f1_scores = []
precision_scores = []
recall_scores = []

for fold, (train_index, val_index) in enumerate(kf.split(X_t)):
    X_train_t, X_val_t = X_t[train_index].to(device), X_t[val_index].to(device)
    y_train_t, y_val_t = y_enc[train_index].to(device), y_enc[val_index].to(device)
    
    model: MLPModel = MLPModel(input_dim, hidden_dim_1, hidden_dim_2, output_dim).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    for epoch in range(300):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train_t).squeeze()
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()
        # print(f"Epoch {epoch}: loss {loss}")
    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_t).squeeze()
        y_pred_t = (torch.sigmoid(val_outputs) > 0.5).long()
        y_pred = y_pred_t.cpu().numpy()
        y_true = y_val_t.cpu().numpy()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    acc_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)
    
print(f"Acurácia: {sum(acc_scores)/num_splits:.4f}")
print(f"Precisão: {sum(precision_scores)/num_splits:.4f}")
print(f"Revocação: {sum(recall_scores)/num_splits:.4f}")
print(f"F1: {sum(f1_scores)/num_splits:.4f}")

Acurácia: 0.7023
Precisão: 0.5206
Revocação: 0.5643
F1: 0.5326


In [7]:
acc = round(sum(acc_scores)/num_splits, 2)

model_path = f"{MODELS_PATH}/modelo_{acc}_{hidden_dim_1}-{hidden_dim_2}.pth"
model_details_path = f"{MODELS_PATH}/modelo_{acc}_{hidden_dim_1}-{hidden_dim_2}.json"

torch.save(model, model_path)

model_details = {
    "optimizer": "Adam",
    "funcao_ativacao": "ReLU",
    "num_epochs": 300,
    "hidden_layer_1": hidden_dim_1,
    "hidden_layer_2": hidden_dim_2,
    "learning_rate": 0.0001
}

with open(model_details_path, 'w') as f:
    json.dump(model_details, f, indent=4)


print(f"Modelo salvo em {model_path}.")
print(f"Detalhes salvos em {model_details_path}")

Modelo salvo em ../models/modelo_0.7_8-10.pth.
Detalhes salvos em ../models/modelo_0.7_8-10.json
